# tpu-retrieval-bench on Colab TPU

Runtime > Change runtime type > TPU v5e-1. Results land in `results/` and are zipped at the end.

In [ ]:
import sys
print(sys.version)
if sys.version_info < (3, 12):
    print('WARNING: python < 3.12. Fallback: pip install "jax[tpu]<0.11" and '
          'pip install --ignore-requires-python -e ".[data,plot]" (repo declares >= 3.12).')

In [ ]:
!git clone https://github.com/maxpetrusenko/tpu-retrieval-bench
%cd tpu-retrieval-bench
!pip install -q -e ".[data,plot]" "jax[tpu]==0.12.0.1" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
# jax pinned to the uv.lock version (0.12.0.1). If pip changed jax, restart the runtime
# (Runtime > Restart session) before running the next cell.

In [ ]:
# Run in a subprocess: importing jax in the notebook kernel would lock the TPU and make every
# later `!python` cell fail with "TPU is already in use".
!python -c "import jax; d = jax.devices(); assert d[0].platform == 'tpu', f'not a TPU runtime: {d}'; print(d[0].device_kind, len(d), 'device(s)')"

## Synthetic sweep

In [ ]:
!python bench/run.py --algos jit_topk,blockwise,pallas_score,approx_max_k,score_only,select_only --m 1,16,128,1024 --n 65536,1000000 --d 128 --k 10 --dtype bf16

## ann-benchmarks SIFT (525 MB)

In [ ]:
!python bench/download.py sift

In [ ]:
!python bench/run.py --dataset sift-128-euclidean --algos jit_topk,blockwise,pallas_score,approx_max_k,score_only,select_only --k 10 --dtype bf16 --n-test 1000 --batch-size 1000

## Optional: Pallas block sweep and approx_max_k recall targets (not in the first run)

In [ ]:
!python bench/run.py --algos pallas_score --sweep-blocks --m 128,1024 --n 1000000 --d 128 --k 10 --dtype bf16


In [ ]:
!for r in 0.90 0.99; do python bench/run.py --algos approx_max_k --recall-target $r --m 1024 --n 1000000 --d 128 --k 10 --dtype bf16; done


## Profiler trace for one config

In [ ]:
# Script, not an in-kernel cell: the kernel must never import jax (it would hold the TPU).
!python bench/profile_one.py --m 128 --n 1000000 --d 128 --k 10 --out results/profile_jit_topk_m128_n1e6_d128

## Plots and download

In [ ]:
!python bench/plot.py

In [ ]:
!zip -qr results.zip results docs/img
from google.colab import files
files.download("results.zip")